In [ ]:
PresentDate = "2021-11-18"

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import scipy.stats as si

In [ ]:
dd1 = yf.download("^GSPC", start="2020-02-01", end=PresentDate)
dd2 = yf.download("^GSPC", start="2020-09-01", end=PresentDate)

In [ ]:
dd1.info()

In [ ]:
dd2.info()

In [ ]:
ln_return1 = np.log(dd2['Adj Close'] / dd2['Adj Close'].shift(1))

In [ ]:
normal_return = []
for i in range(0,len(dd1)-1):
    adjclose_yesterday = dd1.iloc[i]['Adj Close']
    adjclose_today = dd1.iloc[i+1]['Adj Close']
    x = (adjclose_today - adjclose_yesterday) / adjclose_yesterday
    normal_return.append(x)
normal_return[:5]

In [ ]:
ln_return = []
for i in range(0,len(dd1)-1):
    adjclose_yesterday = dd1.iloc[i]['Adj Close']
    adjclose_today = dd1.iloc[i+1]['Adj Close']
    y = np.log(adjclose_today / adjclose_yesterday)
    ln_return.append(y)
ln_return[:5]

In [ ]:
fn = pd.DataFrame(normal_return, columns = ['normal']) 
nr = fn.mean() * len(fn)
nv = fn.std() * (len(fn) ** 0.5)
print('Annulized normal return = %.8f' % nr)
print('Annulized by = %.8f' % nv)

In [ ]:
fig = plt.figure(figsize= (15, 8))
plt.plot(fn * 100) 
plt.xlabel('Days')
plt.ylabel('Percentage % ')
plt.title('Log Return');

In [ ]:
ln_return = np.log(dd1['Adj Close'] / dd1['Adj Close'].shift(1))
v = np.sqrt(252) * ln_return.std()
print(round(v*100,2))

In [ ]:
fig = plt.figure(figsize= (15, 8))
plt.plot(dd1['Adj Close'])
plt.xlabel('Days')
plt.ylabel('Price')
plt.title('Adj Closed Price');

In [ ]:
st, ttm = np.meshgrid(np.array([100, 110, 120, 130, 140]), np.array([6/12, 10/12]))
a2 = np.array([np.array([0.369229563, 0.360583371, 0.355315095, 0.356638788, 0.359157824]), 
               np.array([0.348106828, 0.354150754, 0.345219209, 0.346127746, 0.34955913])])
print(a2)

In [ ]:
fig = plt.figure(figsize= (15, 15))
ax = fig.gca(projection='3d')
surf = ax.plot_surface(st, ttm, a2, rstride=2, cstride=2, cmap=plt.cm.coolwarm, linewidth=0.1, antialiased=True)
ax.set_xlabel('Strike')
ax.set_ylabel('Time to Maturitre')
ax.set_zlabel('Volatile Implied')
fig.colorbar(surf, shrink=0.5, aspect=1,);

In [ ]:
vol = np.sqrt(252) * ln_return1.std()
print(round(vol*100,2))

In [ ]:
T = 1/12
r = 0.0154
sigma = 0.40
S0 = 121.08
K = 120

def mcs_simulation_np(p):
    M = p
    I = p
    dt = T / M 
    S = np.zeros((M + 1, I))
    S[0] = S0 
    rn = np.random.standard_normal(S.shape) 
    for t in range(1, M + 1): 
        S[t] = S[t-1] * np.exp((r - sigma ** 2 / 2) * dt + sigma * np.sqrt(dt) * rn[t]) 
    return S

In [ ]:
S = np.transpose(mcs_simulation_np(900))
print(S)

In [ ]:
fig = plt.figure(figsize= (15, 8))
n, bins, patches = plt.hist(x=S[:,-1], color='green', bins='auto', rwidth=0.5)
plt.grid(alpha=0)
ax = plt.ylabel('Frequency')

In [ ]:
put = (K - S[-1,:])>0
call = (S[-1,:]  - K)>0
px = np.mean(np.maximum(put.astype(int),0))
cx = np.mean(np.maximum(call.astype(int),0))
print(cx, px, cx + px)

In [ ]:
def bollen_and_whaley(S, K, T, r, q, vol, payoff):
    d1 = (np.log(S / K) + (r - q + 0.5 * vol ** 2) * T) / (vol * np.sqrt(T))
    d2 = (np.log(S / K) + (r - q - 0.5 * vol ** 2) * T) / (vol * np.sqrt(T))
    if payoff == "call":
        option_value = np.exp(-q * T) * si.norm.cdf(d2, 0.0, 1.0)
    elif payoff == "put":
        option_value = np.exp(-q * T) * si.norm.cdf(-d2, 0.0, 1.0)    
    return option_value

In [ ]:
def binary_option_bsm(S, K, T, r, q, vol, payoff):    
    d1 = (np.log(S / K) + (r - q + 0.5 * vol ** 2) * T) / (vol * np.sqrt(T))
    d2 = (np.log(S / K) + (r - q - 0.5 * vol ** 2) * T) / (vol * np.sqrt(T))
    if payoff == "call":
        option_value = np.exp(-q * T) * si.norm.cdf(d2, 0.0, 1.0)
    elif payoff == "put":
        option_value = np.exp(-q * T) * si.norm.cdf(-d2, 0.0, 1.0)    
    return option_value

In [ ]:
binary_option_bsm(121.08, 120, 1/12, 0.0154, 0, 0.4, 'put')

In [ ]:
binary_option_bsm(121.08, 120, 1/12, 0.0154, 0, 0.4, 'call')

In [ ]:
(binary_option_bsm(121.08, 120, 1/12, 0.0154, 0, 0.4, 'put')+binary_option_bsm(121.08, 120, 1/12, 0.0154, 0, 0.4, 'call'))

In [ ]:
def delta(S, K, T, r, q, vol, payoff):    
    d1 = (np.log(S / K) + (r - q + 0.5 * vol ** 2) * T) / (vol * np.sqrt(T))
    d2 = (np.log(S / K) + (r - q - 0.5 * vol ** 2) * T) / (vol * np.sqrt(T))
    if payoff == "call":
        delta = np.exp(-r * T) * si.norm.pdf(d2, 0.0, 1.0)/(vol * S * np.sqrt(T))
    elif payoff == "put":
        delta =  - np.exp(-r * T) * si.norm.pdf(d2, 0.0, 1.0)/(vol * S * np.sqrt(T))    
    return delta

print(delta(121.08, 120, 1/12, 0.0154, 0, 0.4, 'call'))
print(delta(121.08, 120, 1/12, 0.0154, 0, 0.4, 'put'))

S = np.linspace(50,150,11)
dc = np.zeros((len(S),1))
dp = np.zeros((len(S),1))
for i in range(len(S)):
    dc [i] = delta(S[i], 120, 1/12, 0.0154, 0, 0.4, 'call')
    dp [i] = delta(S[i], 120, 1/12, 0.0154, 0, 0.4, 'put')

In [ ]:
fig = plt.figure(figsize= (15, 8))
plt.plot(S, dc, '-')
plt.plot(S, dp, '-')
plt.grid()
plt.xlabel('Stock Price')
plt.ylabel('Delta')
plt.title('Delta')
ax = plt.legend(['Delta for Call','Delta for Put'])

In [ ]:
S = np.linspace(50, 150, 51)
T = np.linspace(0.5, 2, 51)
Delta = np.zeros((len(T),len(S)))
for j in range(len(S)):
    for i in range(len(T)):
        Delta[i,j] = delta(S[j], 120, T[i], 0.0154, 0, 0.4, 'put')

In [ ]:
Delta

In [ ]:
fig = plt.figure(figsize= (15, 8))
ax = fig.gca(projection='3d')
S, T = np.meshgrid(S, T)
surf = ax.plot_surface(S, T, Delta, rstride=2, cstride=2, cmap=plt.cm.coolwarm, linewidth=0.5, antialiased=True)
ax.set_xlabel('Stock Price')
ax.set_ylabel('Time to Expiry')
ax.set_zlabel('Delta')
fig.colorbar(surf, shrink=0.5, aspect=5);

In [ ]:
def gamma(S, K, T, r, q, vol, payoff):
    d1 = (np.log(S / K) + (r - q + 0.5 * vol ** 2) * T) / (vol * np.sqrt(T))
    d2 = (np.log(S / K) + (r - q - 0.5 * vol ** 2) * T) / (vol * np.sqrt(T))
    if payoff == "call":
        gamma = -(np.exp(-r * T) * d1* si.norm.pdf(d2, 0.0, 1.0) / (vol**2 * S**2 *(T)))
    elif payoff == "put":
        gamma = (np.exp(-r * T) * d1* si.norm.pdf(d2, 0.0, 1.0) / (vol**2 * S**2 *(T)))
    return gamma

print(gamma(121.08, 120, 1/12, 0.0154, 0, 0.4, 'call'))
print(gamma(121.08, 120, 1/12, 0.0154, 0, 0.4, 'put'))

S = np.linspace(50,150,11)
Gamma = np.zeros((len(S),1))
for i in range(len(S)):
    Gamma[i] = gamma(S[i], 120, 1/12, 0.0154, 0, 0.4, 'put')

In [ ]:
fig = plt.figure(figsize= (15, 8))
plt.plot(S, Gamma, '-')
plt.grid()
plt.xlabel('Stock Price')
plt.ylabel('Gamma')
plt.title('Gamma')
ax = plt.legend(['Gamma for Put'])

In [ ]:
S = np.linspace(50, 150, 51)
T = np.linspace(0.5, 2, 51)
Gamma = np.zeros((len(T),len(S)))
for j in range(len(S)):
    for i in range(len(T)):
        Gamma[i,j] = gamma(S[j], 120, T[i], 0.0154, 0, 0.4, 'put')

In [ ]:
fig = plt.figure(figsize=(15, 10))
ax = fig.gca(projection='3d')
S, T = np.meshgrid(S, T)
surf = ax.plot_surface(S, T, Gamma, rstride=2, cstride=2, cmap=plt.cm.coolwarm, linewidth=0.5, antialiased=True)
ax.set_xlabel('Stock Price')
ax.set_ylabel('Time to Expiry')
ax.set_zlabel('Gamma')
fig.colorbar(surf, shrink=0.5, aspect=5);

In [ ]:
def speed(S, K, T, r, q, vol, payoff):
    d1 = (np.log(S / K) + (r - q + 0.5 * vol ** 2) * T) / (vol * np.sqrt(T))
    d2 = (np.log(S / K) + (r - q - 0.5 * vol ** 2) * T) / (vol * np.sqrt(T))
    if payoff == "call":
        speed =-((np.exp(-r*T)*si.norm.pdf(d2,0.0,1))/(vol**2*S**3*T))*(-2*d1 + (1-d1*d2)/(vol*np.sqrt(T)))
    elif payoff == "put":
        speed =((np.exp(-r*T)*si.norm.pdf(d2,0.0,1))/(vol**2*S**3*T))*(-2*d1 + (1-d1*d2)/(vol*np.sqrt(T)))
    return speed

print(speed(121.08,120,1/12,0.0154,0,0.4,'call'))
print(speed(121.08,120,1/12,0.0154,0,0.4,'put'))

S = np.linspace(80,180,15)
Speed = np.zeros((len(S),1))
for i in range(len(S)):
    Speed [i] = speed(S[i], 100, 1, 0.05, 0.03, 0.25, 'call')

In [ ]:
fig = plt.figure(figsize=(10, 6))
plt.plot(S, Speed, '-')
plt.grid()
plt.xlabel('Stock Price')
plt.ylabel('Speed')
plt.title('Speed')
plt.legend(['Speed for Put'])

In [ ]:
def vega(S, K, T, r, q, vol, payoff):
    d1 = (np.log(S / K) + (r - q + 0.5 * vol ** 2) * T) / (vol * np.sqrt(T))
    d2 = (np.log(S / K) + (r - q - 0.5 * vol ** 2) * T) / (vol * np.sqrt(T))
    if payoff == "call":
        vega = -((np.exp(-r * T) * si.norm.pdf(d2, 0.0, 1.0)) * ((np.sqrt(T)) + (d2/vol)))
    elif payoff == "put":
        vega = (np.exp(-r * T) * si.norm.pdf(d2, 0.0, 1.0)) * ((np.sqrt(T)) + (d2/vol))
    return vega

print(vega(121.08, 120, 1/12, 0.0154, 0, 0.4, 'call'))
print(vega(121.08, 120, 1/12, 0.0154, 0, 0.4, 'put'))


vol = np.linspace(0.1,0.6,13)
Vega = np.zeros((len(vol),1))
for i in range(len(vol)):
    Vega [i] = vega(121.08, 120, 1/12, 0.0154, 0, vol[i], 'call')

In [ ]:
fig = plt.figure(figsize= (15, 8))
plt.plot(vol, Vega, '--', color='Red')
plt.grid()
plt.xlabel('Volatility')
plt.ylabel('Vega')
plt.title('Vega')
plt.legend(['Vega for Put'])

# SCATTERPLOT

In [ ]:
S, T = np.meshgrid(S, T)


In [ ]:
aa = pd.DataFrame(data=T)

In [ ]:
aa

In [ ]:
# start with simple scatterplot
sns.scatterplot(x='Day_till_expiration',y='VKOSPI',data=train_org)
plt.show()

# too bizarre when VKOSPI is small.
print('Too bizarre when VKOSPI is small.')
print('Instead of plotting every data, plot mean of VKOSPI within each Day_till_expiration.')
      
mean_by_dtexp = train_org.groupby('Day_till_expiration').mean()['VKOSPI']
plt.figure(figsize=(15,5))
plt.ylabel('Mean of VKOSPI')
mean_by_dtexp_plot = mean_by_dtexp - 10  # for plotting purpose
sns.barplot(x=mean_by_dtexp.index,y=mean_by_dtexp_plot.values,bottom=10)
plt.show()

# reasonable to suspect that there aren't much data with very high day_till_expiration
print('It is reasonable to suspect that there aren\'t much data when day_till_expiration is high')
len_by_dtexp = train_org['Day_till_expiration'].value_counts()
plt.figure(figsize=(15,5))
plt.xlabel('Day_till_expiration')
plt.ylabel('Frequency of Data')
sns.barplot(x=len_by_dtexp.index,y=len_by_dtexp.values)
plt.show()

print('The amount of data seems to be insufficient when Day_till_expiration is high')